In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
import os
import torch.nn as nn
import torch
from os import path as osp
import math


import sys
import os

import torch.nn as nn
import torch
from os import path as osp
import math

import torchvision.transforms as T

class Encoder_2to1(nn.Module):
    def __init__(self, cin=3, cout=3, in_size=64, nf=64, activation=nn.Tanh):
        super(Encoder_2to1, self).__init__()

        max_channels = 8 * nf
        num_layers = int(math. log2(in_size)) - 1
        channels = [cin] + [min(nf * (2 ** i), max_channels) for i in range(num_layers)]

        self.layers = nn.ModuleList(
            [nn.Sequential(
                nn.Conv2d(channels[i], channels[i+1], kernel_size=4, stride=2, padding=1 if i != num_layers - 1 else 0, bias=False),
                nn.ReLU(inplace=True)
            ) for i in range(num_layers)]
        )
        if activation is not None:
            self.out_layer = nn.Sequential(
                nn.Conv2d(max_channels, cout, kernel_size=1, stride=1, padding=0, bias=False),
                activation()
            )
        else:
            self.out_layer = nn.Conv2d(max_channels, cout, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.out_layer(x).reshape(x.size(0),-1)

    def freeze(self):
        for param in self.parameters():
            param.requires_grad_(False)

    def unfreeze(self):
        for param in self.parameters():
            param.requires_grad_(True)

class DenseBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//4)
        self.conv2 = mobile_unit(channel_in*5//4, channel_in//4)
        self.conv3 = mobile_unit(channel_in*6//4, channel_in//4)
        self.conv4 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        out4 = self.conv4(comb3)
        comb4 = torch.cat((comb3, out4),dim=1)
        return comb4


class DenseBlock2(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//2)
        self.conv2 = mobile_unit(channel_in*3//2, channel_in//2)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        return comb2


class DenseBlock3(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock3, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in)
        self.conv2 = mobile_unit(channel_in*2, channel_in)
        self.conv3 = mobile_unit(channel_in*3, channel_in)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        return comb3


class DenseBlock2_noExpand(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2_noExpand, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in*3//4)
        self.conv2 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((out1, out2),dim=1)
        return comb2


class SenetBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel, size):
        super(SenetBlock, self).__init__()
        self.size = size
        self.globalAvgPool = nn.AdaptiveAvgPool2d((1, 1))
        self.channel = channel
        self.fc1 = linear_layer(self.channel, min(self.channel//2, 256))
        self.fc2 = linear_layer(min(self.channel//2, 256), self.channel, relu=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        original_out = x
        pool = self.globalAvgPool(x)
        pool = pool.view(pool.size(0), -1)
        fc1 = self.fc1(pool)
        out = self.fc2(fc1)
        out = self.sigmoid(out)
        out = out.view(out.size(0), out.size(1), 1, 1)

        return out * original_out


def conv_layer(channel_in, channel_out, ks=1, stride=1, padding=0, dilation=1, bias=False, bn=True, relu=True, group=1):
    """Conv block

    Args:
        channel_in (int): input channel size
        channel_out (int): output channel size
        ks (int, optional): kernel size. Defaults to 1.
        stride (int, optional): Defaults to 1.
        padding (int, optional): Defaults to 0.
        dilation (int, optional): Defaults to 1.
        bias (bool, optional): Defaults to False.
        bn (bool, optional): Defaults to True.
        relu (bool, optional): Defaults to True.
        group (int, optional): group conv parameter. Defaults to 1.

    Returns:
        Sequential: a block with bn and relu
    """
    _conv = nn.Conv2d
    sequence = [_conv(channel_in, channel_out, kernel_size=ks, stride=stride, padding=padding, dilation=dilation,
                      bias=bias, groups=group)]
    if bn:
        sequence.append(nn.BatchNorm2d(channel_out))
    if relu:
        sequence.append(nn.ReLU())

    return nn.Sequential(*sequence)


def linear_layer(channel_in, channel_out, bias=False, bn=True, relu=True):
    """Fully connected block

    Args:
        channel_in (int): input channel size
        channel_out (_type_): output channel size
        bias (bool, optional): Defaults to False.
        bn (bool, optional): Defaults to True.
        relu (bool, optional): Defaults to True.

    Returns:
        Sequential: a block with bn and relu
    """
    _linear = nn.Linear
    sequence = [_linear(channel_in, channel_out, bias=bias)]

    if bn:
        sequence.append(nn.BatchNorm1d(channel_out))
    if relu:
        sequence.append(nn.Hardtanh(0,4))

    return nn.Sequential(*sequence)


class mobile_unit(nn.Module):
    dump_patches = True

    def __init__(self, channel_in, channel_out, stride=1, has_half_out=False, num3x3=1):
        """Init a depth-wise sparable convolution

        Args:
            channel_in (int): input channel size
            channel_out (_type_): output channel size
            stride (int, optional): conv stride. Defaults to 1.
            has_half_out (bool, optional): whether output intermediate result. Defaults to False.
            num3x3 (int, optional): amount of 3x3 conv layer. Defaults to 1.
        """
        super(mobile_unit, self).__init__()
        self.stride = stride
        self.channel_in = channel_in
        self.channel_out = channel_out
        if num3x3 == 1:
            self.conv3x3 = nn.Sequential(
                conv_layer(channel_in, channel_in, ks=3, stride=stride, padding=1, group=channel_in),
            )
        else:
            self.conv3x3 = nn.Sequential(
                conv_layer(channel_in, channel_in, ks=3, stride=1, padding=1, group=channel_in),
                conv_layer(channel_in, channel_in, ks=3, stride=stride, padding=1, group=channel_in),
            )
        self.conv1x1 = conv_layer(channel_in, channel_out)
        self.has_half_out = has_half_out

    def forward(self, x):
        half_out = self.conv3x3(x)
        out = self.conv1x1(half_out)
        if self.stride == 1 and (self.channel_in == self.channel_out):
            out = out + x
        if self.has_half_out:
            return half_out, out
        else:
            return out

class DenseStack(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel):
        super(DenseStack, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2, 32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4,16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4, 4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.dense3(d2))
        u1 = self.upsample1(self.senet4(self.thrink1(d3)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.upsample3(self.senet6(self.thrink3(us2)))
        return u3


class DenseStack2(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel, final_upsample=True, ret_mid=False):
        super(DenseStack2, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2,32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4, 16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4,4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.final_upsample = final_upsample
        if self.final_upsample:
            self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.ret_mid = ret_mid

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.senet3(self.dense3(d2)))
        d4 = self.dense5(self.dense4(d3))
        u1 = self.upsample1(self.senet4(self.thrink1(d4)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.senet6(self.thrink3(us2))
        if self.final_upsample:
            u3 = self.upsample3(u3)
        if self.ret_mid:
            return u3, u2, u1, d4
        else:
            return u3, d4


class DenseStack_Backnone(nn.Module):
    def __init__(self, input_channel=128, out_channel=24, latent_size=256, kpts_num=21, pretrain=False):
        super(DenseStack_Backnone, self).__init__()
        self.pre_layer = nn.Sequential(conv_layer(3, input_channel // 2, 3, 2, 1),
                                       mobile_unit(input_channel // 2, input_channel))
        self.thrink = conv_layer(input_channel * 4, input_channel)
        self.dense_stack1 = DenseStack(input_channel, out_channel)
        self.stack1_remap = conv_layer(out_channel, out_channel)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.thrink2 = conv_layer((out_channel + input_channel), input_channel)
        self.dense_stack2 = DenseStack2(input_channel, out_channel, final_upsample=False)
        self.mid_proj = conv_layer(1024, latent_size, 1, 1, 0, bias=False, bn=False, relu=False)
        self.reduce = conv_layer(out_channel, kpts_num, 1, bn=False, relu=False)
        self.uv_reg = nn.Sequential(linear_layer(latent_size, 128, bn=False),
                                    linear_layer(128, 64, bn=False),
                                    linear_layer(64, 3, bn=False, relu=False))
        self.reorg = conv_layer(input_channel, input_channel * 4, 3, 2, 1) # Reorg()
        if pretrain:
            cur_dir = os.path.dirname(os.path.realpath(__file__))
            weight = torch.load(os.path.join(cur_dir, '../misc/densestack.pth'))
            self.load_state_dict(weight, strict=False)
            print('Load pre-trained weight: densestack.pth')

        self.act = nn.Sigmoid()
    def forward(self, x):
        pre_out = self.pre_layer(x) # [1, 128, 128, 128]

        pre_out_reorg = self.reorg(pre_out)  # [1, 512, 64, 64]
        thrink = self.thrink(pre_out_reorg)

        stack1_out = self.dense_stack1(thrink)
        stack1_out_remap = self.stack1_remap(stack1_out)
        input2 = torch.cat((stack1_out_remap, thrink),dim=1)
        thrink2 = self.thrink2(input2)
        stack2_out, stack2_mid = self.dense_stack2(thrink2)

        latent = self.mid_proj(stack2_mid)
        # import pdb; pdb.set_trace()
        uv_reg = self.uv_reg(self.reduce(stack2_out).view(stack2_out.shape[0], 21, -1))

        return latent, self.act(uv_reg)



class MobRecon_DS_moonchaeboo(nn.Module):
    def __init__(self, cfg):
        super(MobRecon_DS_moonchaeboo, self).__init__()
        self.latent_size = 1024 # 256 --> 1024   240 --> 900
        self.backbone = DenseStack_Backnone(latent_size=self.latent_size, kpts_num=21)
        self.decoder3d = Joint3DDecoder(self.latent_size, uv_channels=128, out_channels=[128, 256, 512]) # 256

    def forward(self, x):
        latent, pred25d = self.backbone(x)
        pred3d = self.decoder3d(pred25d, latent)

        return pred3d,


class Encoder_2to1(nn.Module):
    def __init__(self, cin=3, cout=3, in_size=64, nf=64, activation=nn.Tanh):
        super(Encoder_2to1, self).__init__()

        max_channels = 8 * nf
        num_layers = int(math. log2(in_size)) - 1
        channels = [cin] + [min(nf * (2 ** i), max_channels) for i in range(num_layers)]

        self.layers = nn.ModuleList(
            [nn.Sequential(
                nn.Conv2d(channels[i], channels[i+1], kernel_size=4, stride=2, padding=1 if i != num_layers - 1 else 0, bias=False),
                nn.ReLU(inplace=True)
            ) for i in range(num_layers)]
        )
        if activation is not None:
            self.out_layer = nn.Sequential(
                nn.Conv2d(max_channels, cout, kernel_size=1, stride=1, padding=0, bias=False),
                activation()
            )
        else:
            self.out_layer = nn.Conv2d(max_channels, cout, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.out_layer(x).reshape(x.size(0),-1)

    def freeze(self):
        for param in self.parameters():
            param.requires_grad_(False)

    def unfreeze(self):
        for param in self.parameters():
            param.requires_grad_(True)

class DenseBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//4)
        self.conv2 = mobile_unit(channel_in*5//4, channel_in//4)
        self.conv3 = mobile_unit(channel_in*6//4, channel_in//4)
        self.conv4 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        out4 = self.conv4(comb3)
        comb4 = torch.cat((comb3, out4),dim=1)
        return comb4


class DenseBlock2(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//2)
        self.conv2 = mobile_unit(channel_in*3//2, channel_in//2)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        return comb2


class DenseBlock3(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock3, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in)
        self.conv2 = mobile_unit(channel_in*2, channel_in)
        self.conv3 = mobile_unit(channel_in*3, channel_in)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        return comb3


class DenseBlock2_noExpand(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2_noExpand, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in*3//4)
        self.conv2 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((out1, out2),dim=1)
        return comb2


class SenetBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel, size):
        super(SenetBlock, self).__init__()
        self.size = size
        self.globalAvgPool = nn.AdaptiveAvgPool2d((1, 1))
        self.channel = channel
        self.fc1 = linear_layer(self.channel, min(self.channel//2, 256))
        self.fc2 = linear_layer(min(self.channel//2, 256), self.channel, relu=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        original_out = x
        pool = self.globalAvgPool(x)
        pool = pool.view(pool.size(0), -1)
        fc1 = self.fc1(pool)
        out = self.fc2(fc1)
        out = self.sigmoid(out)
        out = out.view(out.size(0), out.size(1), 1, 1)

        return out * original_out


class DenseStack(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel):
        super(DenseStack, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2, 32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4,16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4, 4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.dense3(d2))
        u1 = self.upsample1(self.senet4(self.thrink1(d3)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.upsample3(self.senet6(self.thrink3(us2)))
        return u3


class DenseStack2(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel, final_upsample=True, ret_mid=False):
        super(DenseStack2, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2,32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4, 16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4,4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.final_upsample = final_upsample
        if self.final_upsample:
            self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.ret_mid = ret_mid

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.senet3(self.dense3(d2)))
        d4 = self.dense5(self.dense4(d3))
        u1 = self.upsample1(self.senet4(self.thrink1(d4)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.senet6(self.thrink3(us2))
        if self.final_upsample:
            u3 = self.upsample3(u3)
        if self.ret_mid:
            return u3, u2, u1, d4
        else:
            return u3, d4


class DenseStack_Backnone(nn.Module):
    def __init__(self, input_channel=128, out_channel=24, latent_size=256, kpts_num=21, pretrain=False):
        super(DenseStack_Backnone, self).__init__()
        self.pre_layer = nn.Sequential(conv_layer(3, input_channel // 2, 3, 2, 1),
                                       mobile_unit(input_channel // 2, input_channel))
        self.thrink = conv_layer(input_channel * 4, input_channel)
        self.dense_stack1 = DenseStack(input_channel, out_channel)
        self.stack1_remap = conv_layer(out_channel, out_channel)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.thrink2 = conv_layer((out_channel + input_channel), input_channel)
        self.dense_stack2 = DenseStack2(input_channel, out_channel, final_upsample=False)
        self.mid_proj = conv_layer(1024, latent_size, 1, 1, 0, bias=False, bn=False, relu=False)
        self.reduce = conv_layer(out_channel, kpts_num, 1, bn=False, relu=False)
        self.uv_reg = nn.Sequential(linear_layer(latent_size, 128, bn=False),
                                    linear_layer(128, 64, bn=False),
                                    linear_layer(64, 3, bn=False, relu=False))
        self.reorg = conv_layer(input_channel, input_channel * 4, 3, 2, 1) # Reorg()
        if pretrain:
            cur_dir = os.path.dirname(os.path.realpath(__file__))
            weight = torch.load(os.path.join(cur_dir, '../misc/densestack.pth'))
            self.load_state_dict(weight, strict=False)
            print('Load pre-trained weight: densestack.pth')

        self.act = nn.Sigmoid()
    def forward(self, x):
        pre_out = self.pre_layer(x) # [1, 128, 128, 128]

        pre_out_reorg = self.reorg(pre_out)  # [1, 512, 64, 64]
        thrink = self.thrink(pre_out_reorg)

        stack1_out = self.dense_stack1(thrink)
        stack1_out_remap = self.stack1_remap(stack1_out)
        input2 = torch.cat((stack1_out_remap, thrink),dim=1)
        thrink2 = self.thrink2(input2)
        stack2_out, stack2_mid = self.dense_stack2(thrink2)

        latent = self.mid_proj(stack2_mid)
        # import pdb; pdb.set_trace()
        uv_reg = self.uv_reg(self.reduce(stack2_out).view(stack2_out.shape[0], 21, -1))

        return latent, self.act(uv_reg)




# Advanced modules
class Joint3DDecoder(nn.Module):
    def __init__(self, latent_size, uv_channels, out_channels):
        """Init a 3D decoding with sprial convolution

        Args:
            latent_size (int): feature dim of backbone feature
            out_channels (list): feature dim of each spiral layer
            spiral_indices (list): neighbourhood of each hand vertex
            up_transform (list): upsampling matrix of each hand mesh level
            uv_channel (int): amount of 2D landmark
            meshconv (optional): conv method, supporting SpiralConv, DSConv. Defaults to SpiralConv.
        """
        super(Joint3DDecoder, self).__init__()
        self.latent_size = latent_size
        self.out_channels = out_channels
        self.uv_channels = uv_channels

        self.de_layer_conv = conv_layer(self.latent_size, self.out_channels[- 1], 1,
        bn=False, relu=False)
        self.uv_linear = nn.Linear(3, self.uv_channels)
        self.upsample_1 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_2 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_3 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        print("out channels ",  out_channels)
        self.cat_conv1 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1, bias=False)
        self.cat_conv2 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1, bias=False)
        self.cat_conv3 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1, bias=False)
        self.cat_act = nn.ReLU()
        self.cat_conv_final = nn.Conv1d(out_channels[-1] * 3, out_channels[-1], 1, 1, bias=False)

        self.final_conv = nn.ModuleList([])
        for step in range(len(out_channels) - 1):
            self.final_conv.append(
                nn.ModuleList([
                    nn.Conv1d(out_channels[-1 - step], out_channels[-2 - step], 1, 1, bias=False),
                    nn.ReLU(),
                    nn.Conv1d(out_channels[-2 - step], out_channels[-2 - step], 1, 1, bias=False),
                    #nn.GroupNorm(8, out_channels[-2 - step]),
                    #SelfAttn(out_channels[-2 - step]),
                ])
            )

        self.final_linear1 = nn.Sequential(
            nn.Linear(out_channels[0], 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def index(self, uv, feat):
        # uv = uv.unsqueeze(2)  # [B, N, 1, 2]
        uv = uv.reshape(1, -1 , 1, 2)
        samples = torch.zeros((1, 512, 21)).to(uv.device)

        return samples



    def forward(self, uv, x):
        x = self.de_layer_conv(x)
        x_1 = self.index(uv[..., :2], x)
        x_2 = self.index(uv[..., 0::2], x)
        x_3 = self.index(uv[..., 1:], x)

        uv_feat = self.uv_linear(uv).permute(0, 2, 1) # [B, 21, 64]?
        #print(x_1.shape)
        #print(uv_feat.shape)

        x_1 = torch.bmm(x_1, self.upsample_1.repeat(x.size(0), 1, 1).to(x.device))
        x_2 = torch.bmm(x_2, self.upsample_2.repeat(x.size(0), 1, 1).to(x.device))
        x_3 = torch.bmm(x_3, self.upsample_3.repeat(x.size(0), 1, 1).to(x.device))
        #print(x_1.shape)

        x_1 = torch.cat([x_1, uv_feat], dim=1)
        #print(x_1.shape)

        x_1 = self.cat_conv1(x_1)
        #print(x_1.shape)

        x_2 = torch.cat([x_2, uv_feat], dim=1)
        x_2 = self.cat_conv2(x_2)
        x_3 = torch.cat([x_3, uv_feat], dim=1)
        x_3 = self.cat_conv3(x_3)
        x =  torch.cat([x_1, x_2, x_3], dim=1)

        x = self.cat_conv_final(x)
        x = self.cat_act(x)


        # import pdb; pdb.set_trace()
        for i, (conv, act, conv2,) in enumerate(self.final_conv):
            x = conv(x)
            x = act(x)
            x = conv2(x)
            #x = norm(x)
            # x = attn(x) + x

        x = x.permute(0, 2, 1)
        x = self.final_linear1(x)
        return x

class MobRecon_DS(nn.Module):
    def __init__(self,):
        super(MobRecon_DS, self).__init__()
        self.latent_size = 1024 # 256 --> 1024   240 --> 900
        self.backbone = DenseStack_Backnone(latent_size=self.latent_size, kpts_num=21)

    def forward(self, x):
        _, pred25d = self.backbone(x)

        return pred25d

In [3]:
!pip install onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 98.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 110.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.0 MB/s eta 0:00:00


In [11]:
model = MobRecon_DS()
pth_path = '/content/drive/MyDrive/KCVL/etri/Large_model_input256.pth'
model.load_state_dict(torch.load(pth_path), strict=False)
model.eval()

MobRecon_DS(
  (backbone): DenseStack_Backnone(
    (pre_layer): Sequential(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
      (1): mobile_unit(
        (conv3x3): Sequential(
          (0): Sequential(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
            (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU()
          )
        )
        (conv1x1): Sequential(
          (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
        )
      )
    )
    (thrink): Sequential(
      (0): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      

In [12]:
input_image = torch.ones((1,3,256,256))
b = model(input_image)
b

tensor([[[6.6288e-01, 6.4154e-01, 1.6364e-20],
         [5.9128e-01, 5.9077e-01, 0.0000e+00],
         [5.0866e-01, 5.4755e-01, 4.4467e-26],
         [4.7385e-01, 5.1825e-01, 3.9877e-12],
         [4.5407e-01, 5.2383e-01, 1.2355e-12],
         [5.0159e-01, 5.3779e-01, 2.0523e-10],
         [4.3795e-01, 5.2901e-01, 1.6321e-11],
         [4.2163e-01, 5.3143e-01, 1.0335e-11],
         [4.0221e-01, 5.3529e-01, 1.1577e-11],
         [5.1292e-01, 5.4682e-01, 2.7147e-11],
         [4.4233e-01, 5.4155e-01, 2.7280e-11],
         [4.2475e-01, 5.4388e-01, 2.0732e-13],
         [3.9354e-01, 5.7035e-01, 1.1704e-17],
         [5.3955e-01, 5.6072e-01, 4.9935e-12],
         [4.7242e-01, 5.8094e-01, 1.3678e-10],
         [4.5362e-01, 5.9621e-01, 2.0409e-11],
         [4.3055e-01, 5.9780e-01, 8.6594e-19],
         [5.5631e-01, 5.7085e-01, 3.4289e-12],
         [4.9061e-01, 6.0294e-01, 4.3866e-13],
         [4.7703e-01, 6.1496e-01, 3.8759e-11],
         [4.6451e-01, 6.0855e-01, 2.9400e-13]]], grad_fn=<Si

In [14]:
input_image = torch.zeros((1,3,256,256))
torch.onnx._export(model, input_image, "/content/drive/MyDrive/KCVL/etri/mob_256_final2.onnx", export_params=True, verbose=False, input_names=['input0'], output_names=['output0'])

<ipython-input-14-f0a924d2d5e2>:2: FutureWarning: 'torch.onnx._export' is deprecated in version 1.12.0 and will be removed in 2.0. Please use `torch.onnx.export` instead.
  torch.onnx._export(model, input_image, "/content/drive/MyDrive/KCVL/etri/mob_256_final2.onnx", export_params=True, verbose=False, input_names=['input0'], output_names=['output0'])


tensor([[[6.3854e-01, 6.1299e-01, 3.9642e-17],
         [6.0019e-01, 5.7455e-01, 0.0000e+00],
         [5.3575e-01, 5.3184e-01, 2.1843e-26],
         [5.0291e-01, 5.0937e-01, 6.1994e-11],
         [4.8470e-01, 5.0046e-01, 4.9110e-11],
         [5.3904e-01, 5.2721e-01, 1.1550e-10],
         [4.7600e-01, 4.9585e-01, 4.8310e-11],
         [4.6203e-01, 4.7772e-01, 9.1773e-12],
         [4.4036e-01, 4.5322e-01, 2.1144e-11],
         [5.2624e-01, 5.2127e-01, 6.6682e-11],
         [4.6332e-01, 4.8299e-01, 1.4253e-10],
         [4.4983e-01, 4.6539e-01, 2.4913e-12],
         [4.3394e-01, 4.6195e-01, 8.0990e-14],
         [5.2868e-01, 5.1922e-01, 2.9440e-11],
         [4.5777e-01, 4.8517e-01, 2.9066e-11],
         [4.5185e-01, 4.7965e-01, 1.6226e-12],
         [4.4249e-01, 4.7231e-01, 1.4098e-14],
         [5.2920e-01, 5.1280e-01, 5.6698e-11],
         [4.5054e-01, 4.8957e-01, 7.9131e-11],
         [4.4833e-01, 4.8726e-01, 1.0957e-12],
         [4.4162e-01, 4.7426e-01, 2.0231e-12]]], grad_fn=<Si